# Module 01 — Pandas Deep Dive (the language of tabular data)

Pandas is where 70–80% of a data scientist's day actually happens. If you're
fluent here, cleaning and EDA feel effortless; if not, everything is a struggle.

We build a precise mental model, then drill the operations you'll use every day:
**selecting, filtering, grouping, joining, reshaping**, and the crucial
`apply`-vs-vectorized decision.

Interview goals for this module:
- Explain **Series vs DataFrame** and what the **index** really is.
- Know **`.loc` vs `.iloc`** cold (and why chained indexing is dangerous).
- Do a **groupby** and explain the *split–apply–combine* pattern.
- Choose the right **merge/join** and predict its row count.
- Reshape with **pivot / melt** and say when each is needed.

## 1.1 The two core objects

- **Series** = one column: a 1-D array **plus a labeled index**.
- **DataFrame** = a dict of Series sharing one index: a 2-D labeled table.

The **index** is not just row numbers — it's a set of *labels* pandas uses to
align data automatically. This alignment is a superpower (and occasionally a
surprise).

In [1]:
import numpy as np
import pandas as pd

s = pd.Series([10, 20, 30], index=["a", "b", "c"], name="score")
print(s)
print("\nindex:", list(s.index))
print("values:", s.values, "  <- underlying NumPy array")

a    10
b    20
c    30
Name: score, dtype: int64

index: ['a', 'b', 'c']
values: [10 20 30]   <- underlying NumPy array


In [2]:
# Automatic alignment by index label (not by position!)
s1 = pd.Series({"a": 1, "b": 2, "c": 3})
s2 = pd.Series({"b": 10, "c": 20, "d": 30})
print(s1 + s2)   # 'a' and 'd' are NaN because they don't exist in both

a     NaN
b    12.0
c    23.0
d     NaN
dtype: float64


**Takeaway:** arithmetic aligns on the index. Mismatched labels produce `NaN`.
This is why a merge/concat gone wrong often shows up as unexpected `NaN`s.

## 1.2 Load the real (messy) dataset

We use `data/customers.csv`, which we deliberately made messy. Here we only
*explore* it — cleaning gets its own module.

In [3]:
df = pd.read_csv("../data/customers.csv")
print("shape:", df.shape)
df.head()

shape: (508, 9)


,customer_id,age,income,city,plan,tenure_months,monthly_spend,support_calls,churn
0,1,38.0,5000000.00,Kisumu,Basic,14,186.65,1,0
1,2,25.0,76695.05,Nakuru,Basic,46,76.83,1,0
2,3,NaN,34209.54,Nairobi,Basic,29,37.36,1,1
3,4,44.0,23126.82,Mombasa,Premium,25,23.34,0,0
4,5,18.0,11133.74,NAIROBI,Basic,50,12.17,4,1


In [4]:
# The 3 commands you run on EVERY new dataset:
df.info()          # dtypes + non-null counts -> spot missing & wrong types

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    508 non-null    int64  
 1   age            488 non-null    float64
 2   income         485 non-null    float64
 3   city           508 non-null    object 
 4   plan           508 non-null    object 
 5   tenure_months  508 non-null    int64  
 6   monthly_spend  508 non-null    float64
 7   support_calls  508 non-null    int64  
 8   churn          508 non-null    int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 35.8+ KB


In [5]:
df.describe()      # summary stats for numeric columns -> spot outliers/scale

,customer_id,age,income,tenure_months,monthly_spend,support_calls,churn
count,508.000000,488.000000,4.850000e+02,508.000000,508.000000,508.000000,508.000000
mean,250.631890,35.016393,6.464815e+04,34.931102,68.869941,1.484252,0.316929
std,145.013243,9.407921,2.263545e+05,20.444401,46.850177,1.258403,0.465738
min,1.000000,18.000000,7.909290e+03,1.000000,5.840000,0.000000,0.000000
25%,124.750000,28.000000,3.420954e+04,17.000000,38.317500,1.000000,0.000000
50%,251.500000,35.000000,4.930228e+04,33.000000,56.575000,1.000000,0.000000
75%,375.250000,41.000000,6.723779e+04,53.000000,85.825000,2.000000,1.000000
max,500.000000,64.000000,5.000000e+06,71.000000,427.550000,7.000000,1.000000


In [6]:
df.describe(include="object")   # for text/categorical columns

,city,plan
count,508,508
unique,14,3
top,Nairobi,Basic
freq,189,252


Already we can *read the story*: `income` has fewer non-null values (missing
data), and its `max` in `describe()` is wildly larger than the 75% quantile
(that planted outlier). This is the habit — **let the summaries talk to you.**

## 1.3 Selecting columns and rows: `.loc` vs `.iloc`

- `df["col"]` → a Series (one column).
- `df[["a", "b"]]` → a DataFrame (list of columns).
- **`.loc[row_label, col_label]`** → selection by **label**.
- **`.iloc[row_pos, col_pos]`** → selection by **integer position**.

In [7]:
print(type(df["age"]))          # Series
print(type(df[["age", "city"]]))# DataFrame

# label-based
print("\n.loc first row, chosen cols:")
print(df.loc[0, ["age", "plan", "churn"]])

# position-based
print("\n.iloc rows 0-2, first 3 cols:")
print(df.iloc[0:3, 0:3])

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>

.loc first row, chosen cols:
age       38.0
plan     Basic
churn        0
Name: 0, dtype: object

.iloc rows 0-2, first 3 cols:
   customer_id   age      income
0            1  38.0  5000000.00
1            2  25.0    76695.05
2            3   NaN    34209.54


**Gotcha — chained indexing:** `df[df.age > 30]["income"] = 0` may fail silently
(`SettingWithCopyWarning`) because you're assigning into a *temporary copy*.
Always assign through a single `.loc`:

```python
df.loc[df.age > 30, "income"] = 0   # correct, unambiguous
```

## 1.4 Filtering with boolean masks (same idea as NumPy)

In [8]:
premium = df[df["plan"] == "Premium"]
print("Premium customers:", len(premium))

# multiple conditions -> parentheses + & | ~
high_risk = df[(df["support_calls"] >= 3) & (df["tenure_months"] < 12)]
print("High-risk (>=3 calls, <12 months):", len(high_risk))

# isin for membership; ~ to negate
coastal = df[df["city"].str.strip().str.title().isin(["Mombasa"])]
print("Mombasa (after cleaning text):", len(coastal))

Premium customers: 95


High-risk (>=3 calls, <12 months): 10
Mombasa (after cleaning text): 102


## 1.5 Creating & transforming columns — vectorized first

Prefer **vectorized** operations and `np.where` / `.map` over `.apply` with a
Python function, because vectorized runs in C (fast) while `.apply` loops in
Python (slow). Use `.apply` only when there's no vectorized alternative.

In [9]:
# vectorized arithmetic
df["spend_per_month_ratio"] = df["monthly_spend"] / (df["tenure_months"] + 1)

# conditional column WITHOUT a loop
df["risk_flag"] = np.where(df["support_calls"] >= 3, "high", "normal")

# map categories to numbers
plan_rank = {"Basic": 0, "Standard": 1, "Premium": 2}
df["plan_rank"] = df["plan"].map(plan_rank)

df[["monthly_spend", "tenure_months", "spend_per_month_ratio",
    "support_calls", "risk_flag", "plan", "plan_rank"]].head()

,monthly_spend,tenure_months,spend_per_month_ratio,support_calls,risk_flag,plan,plan_rank
0,186.65,14,12.443333,1,normal,Basic,0
1,76.83,46,1.634681,1,normal,Basic,0
2,37.36,29,1.245333,1,normal,Basic,0
3,23.34,25,0.897692,0,normal,Premium,2
4,12.17,50,0.238627,4,high,Basic,0


### When `.apply` is justified

In [10]:
# A rule with no clean vectorized form -> apply on a row (axis=1).
def segment(row):
    if row["plan"] == "Premium" and row["support_calls"] == 0:
        return "loyal_premium"
    if row["support_calls"] >= 3:
        return "at_risk"
    return "standard"

df["segment"] = df.apply(segment, axis=1)
print(df["segment"].value_counts())

segment
standard         396
at_risk           92
loyal_premium     20
Name: count, dtype: int64


## 1.6 GroupBy — the *split–apply–combine* pattern

This is the beating heart of analysis. Pandas **splits** rows into groups,
**applies** a function to each group, then **combines** the results.

In [11]:
# Average monthly spend and churn rate per plan
summary = (
    df.groupby("plan")
      .agg(customers=("customer_id", "count"),
           avg_spend=("monthly_spend", "mean"),
           churn_rate=("churn", "mean"))
      .sort_values("churn_rate", ascending=False)
)
summary.round(3)

,customers,avg_spend,churn_rate
plan,,,
Basic,252,68.167,0.409
Premium,95,63.584,0.242
Standard,161,73.089,0.217


Read it like a sentence: *"Group by plan; for each plan count customers, average
the spend, and average churn (which, since churn is 0/1, IS the churn rate)."*
The trick `mean of a 0/1 column = proportion of 1s` is worth memorizing.

In [12]:
# Group by two keys -> a nice cross-table of churn rate
two_way = df.groupby(["plan", "risk_flag"])["churn"].mean().unstack().round(3)
two_way

risk_flag,high,normal
plan,,
Basic,0.653,0.350
Premium,0.400,0.212
Standard,0.357,0.188


## 1.7 Joining tables: merge

Real projects have many tables. `merge` is the SQL `JOIN` of pandas. The two
things to always ask: **on which key?** and **which join type?**

- `inner` — only matching keys (default). Rows can *shrink*.
- `left` — keep all left rows; unmatched right side = NaN.
- `right`, `outer` — the mirror / union.

**Predict the row count before you run it** — a good habit that catches bugs.

In [13]:
# Build a small lookup table: a discount per city
discounts = pd.DataFrame({
    "city": ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
    "discount_pct": [5, 8, 6, 4, 7],
})

# Clean the city text first so keys match (we'll formalize this in Module 02)
df["city"] = df["city"].str.strip().str.title()

merged = df.merge(discounts, on="city", how="left")
print("rows before:", len(df), " rows after left-merge:", len(merged))
merged[["city", "discount_pct"]].drop_duplicates().sort_values("city")

rows before: 508  rows after left-merge: 508


,city,discount_pct
13,Eldoret,7
0,Kisumu,6
3,Mombasa,8
2,Nairobi,5
1,Nakuru,4


**Takeaway:** a `left` merge should not change your row count *unless* the right
table has duplicate keys (then rows multiply). Checking `len()` before/after is a
cheap safeguard interviewers love to hear you mention.

## 1.8 Reshaping: wide ↔ long (pivot / melt)

- **Wide**: one row per entity, many columns (human-friendly, good for reports).
- **Long / tidy**: one row per observation (model- and plot-friendly).

`pivot_table` goes long→wide (and can aggregate). `melt` goes wide→long.

In [14]:
wide = df.pivot_table(index="plan", columns="risk_flag",
                      values="monthly_spend", aggfunc="mean").round(1)
print("WIDE (avg spend by plan x risk):")
print(wide)

long = wide.reset_index().melt(id_vars="plan",
                               var_name="risk_flag", value_name="avg_spend")
print("\nLONG (tidy) form:")
print(long)

WIDE (avg spend by plan x risk):
risk_flag  high  normal
plan                   
Basic      64.3    69.1
Premium    68.7    62.6
Standard   69.2    73.9

LONG (tidy) form:
       plan risk_flag  avg_spend
0     Basic      high       64.3
1   Premium      high       68.7
2  Standard      high       69.2
3     Basic    normal       69.1
4   Premium    normal       62.6
5  Standard    normal       73.9


## 1.9 Sorting, ranking, and quick value counts

In [15]:
print("Top 5 spenders:")
print(df.nlargest(5, "monthly_spend")[["customer_id", "plan", "monthly_spend"]])

print("\nCity distribution:")
print(df["city"].value_counts(normalize=True).round(3))  # proportions

Top 5 spenders:
     customer_id      plan  monthly_spend
346          347  Standard         427.55
53            54  Standard         244.80
6              7     Basic         242.23
253          254     Basic         237.17
150          151   Premium         236.80

City distribution:
city
Nairobi    0.413
Mombasa    0.201
Nakuru     0.144
Kisumu     0.138
Eldoret    0.104
Name: proportion, dtype: float64


## 1.10 Mini-exercises

1. Compute the **average income per city**, ignoring missing values. Which city
   is highest? (Hint: `groupby(...).income.mean()`.)
2. Create a column `is_loyal` = 1 if `tenure_months > 36` else 0, **vectorized**.
3. Make a pivot table of **churn rate by city × plan**. Which cell is worst?
4. Explain, in your own words, the difference between `.loc` and `.iloc`, and
   why chained assignment is risky.

In [16]:
# Scratch space
print(df.groupby("city")["income"].mean().round(0).sort_values(ascending=False))
df["is_loyal"] = (df["tenure_months"] > 36).astype(int)
print("\nloyal customers:", df["is_loyal"].sum())

city
Kisumu     128162.0
Eldoret     57459.0
Nakuru      54736.0
Nairobi     54104.0
Mombasa     53486.0
Name: income, dtype: float64

loyal customers: 232


## Summary — pandas fluency checklist

- Series/DataFrame are **labeled** arrays; the **index aligns** data automatically.
- `.info()`, `.describe()`, `.head()` are your first three moves on any dataset.
- `.loc` = labels, `.iloc` = positions; assign via a **single `.loc`**.
- Filter with boolean masks (`&`, `|`, `~`, parentheses).
- Prefer **vectorized / `np.where` / `.map`** over `.apply`; use `.apply` only when forced.
- **GroupBy = split–apply–combine**; `mean` of a 0/1 column is a rate.
- Choose the right **merge** type and **check row counts**.
- **pivot** = long→wide, **melt** = wide→long (tidy).

Next: **Module 02 — Data Cleaning**, where we fix everything wrong with this data.